# This is where we can train our own model

In [53]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

Load the data for training

In [54]:
df = pd.read_csv("../data/train_images.csv")

In [55]:
class BirdDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

        # Convert labels to numeric if needed
        self.classes = sorted(self.df['label'].unique())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open('../data' + row["image_path"]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = self.class_to_idx[row["label"]]
        return img, label


In [56]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

In [57]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(50),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train_dataset = BirdDataset(train_df, transform=train_tfms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [58]:
classes = train_df['label'].unique()
classes

array([ 42, 193,  23,  49,  85,   5,  69, 179,  60,  56,  20,  12,  67,
        65,  84,  70, 186,  38,  21, 124,  17,  86, 147,  33,  77,  68,
       187,   6,  87,  61,  30, 199,  99,  31,  94,  45, 136, 146, 118,
        72,  14, 113,  59, 108, 126,   1, 200, 181, 166,  53, 185,  22,
        62,  16,  48, 172,  15, 125,  75, 140, 153, 133, 103,   9, 112,
       127, 131,  95,  13, 145,   7, 139, 101,  78,  88, 100,  11,  32,
        10,  35,  50, 169,  34, 178, 157, 142,  76,  58,  63,  93,  52,
         2,   3,  74, 110, 141,  64,  29, 120,  46,  97,  83,  57,  39,
        36,  40, 190, 188,  80, 130, 128, 105, 107,  71, 135,  91, 159,
        92,   4, 161, 163, 122,  47,   8,  43,  96,  25, 170,  79, 175,
        73,  19, 121, 149, 137,  44, 150,  55, 111, 194, 114,  90, 115,
        51,  89, 154, 152, 148, 155, 104, 123,  27, 109, 134,  24,  81,
       177,  26, 174, 164,  41,  54, 167, 184,  28, 158, 132, 119, 156,
       195,  66, 162,  18, 173, 117, 182, 160,  37, 191, 143, 19

Train the model

In [59]:
class ImageEmbeddingModel(nn.Module):
    def __init__(self, img_size=224):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 112x112
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 56x56
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 28x28
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),# 14x14

            # ⭐ Added Layer 1
            nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 7x7

            # ⭐ Added Layer 2
            nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 3x3
        )

        # Compute flat size dynamically
        with torch.no_grad():
            x = torch.randn(1, 3, img_size, img_size)
            x = self.backbone(x)
            flat_size = x.numel() // x.shape[0]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 512),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 200)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))


net = ImageEmbeddingModel()


In [60]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.0003)

In [91]:
def get_validation_accuracy():
    val_dataset = BirdDataset(val_df, transform=test_tfms)
    test_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    val_preds = []
    val_labels = []
    net.eval()

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = net(inputs)
            _, predicted = torch.max(outputs, 1)

            val_preds.extend(predicted.numpy())
            val_labels.extend(labels.numpy())

    accuracy = accuracy_score(val_labels, val_preds)
    return accuracy

In [80]:
for epoch in range(60):  # loop over the dataset multiple times
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()

    print(f'[{epoch + 1}] loss: {running_loss/len(train_loader):.3f}')

    running_loss = 0.0

print('Finished Training')

[1] loss: 2.114
[2] loss: 1.963
[3] loss: 1.869
[4] loss: 1.785
[5] loss: 1.656
[6] loss: 1.520
[7] loss: 1.387
[8] loss: 1.274
[9] loss: 1.203
[10] loss: 1.133
[11] loss: 0.986
[12] loss: 0.952
[13] loss: 0.859
[14] loss: 0.814
[15] loss: 0.725
[16] loss: 0.736
[17] loss: 0.668
[18] loss: 0.601
[19] loss: 0.598
[20] loss: 0.589
[21] loss: 0.496
[22] loss: 0.479
[23] loss: 0.433
[24] loss: 0.405
[25] loss: 0.422
[26] loss: 0.330
[27] loss: 0.341
[28] loss: 0.308
[29] loss: 0.329
[30] loss: 0.313
Finished Training


Run the model on the val dataset

In [81]:
idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}

In [82]:
val = get_validation_accuracy()
print(val)

0.2035623409669211


Run the model on test data

In [83]:
test_df = pd.read_csv("../data/test_images_path.csv")

In [84]:
test_dataset = BirdDataset(test_df, transform=test_tfms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [85]:
net.eval()

ImageEmbeddingModel(
  (backbone): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (15): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding

In [86]:
all_ids = test_df["id"].tolist()
all_preds = []

In [87]:
with torch.no_grad():
    for inputs, _ in test_loader:
        outputs = net(inputs)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.numpy())

In [88]:
predicted_labels = [idx_to_class[i] for i in all_preds]

In [89]:
len(predicted_labels)


4000

In [90]:
output_df = pd.DataFrame({
    "id": all_ids,
    "label": predicted_labels
})

output_df.to_csv("test_predictions.csv", index=False)
print("Saved test_predictions.csv!")

Saved test_predictions.csv!
